# Voice Enhancer · M3 training on Kaggle (fine-tune GTCRN)
**One-time setup (in the Kaggle website):**
1. Phone-verify your account (Kaggle requires it to enable GPU and Internet). Create a **New Notebook**, import this file (File > Import Notebook).
2. Right-hand **Settings**: Accelerator = **GPU T4 x2** (or P100; one GPU is used), **Internet = On**. Persistence = "Files only" is handy but not required.
3. **Datasets > New Dataset**: upload `voice-enhancer-m3c.zip` (or newer), keep it **Private**, then in the notebook: **Add Input > Your Datasets > that dataset**.
4. Optional: another private dataset with your test clips (mp3/wav/mp4), added the same way. They are enhanced and benchmarked at the end.

**Running:** use *Run All* to test, then **Save Version > Save & Run All (Commit)** with the GPU to train in the background (you can close the browser).
Kaggle sessions are limited (about 12 h, plus a weekly GPU quota shown in Settings), so the training cell stops itself before the limit and saves a checkpoint every epoch.

**Continuing in a later session:** in the new notebook version, **Add Input > Notebook Output Files > this notebook**. The resume cell finds `last.pt` automatically.

If a cell fails, copy me the error text.

In [ ]:
import os, glob, shutil, subprocess, sys, time
import torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU: check Settings > Accelerator)")
print("cpus:", os.cpu_count(), "| ffmpeg:", shutil.which("ffmpeg"))

# 1) get the code out of the (read-only) input dataset
src = next((r for r, d, f in os.walk("/kaggle/input") if "train.py" in f and os.path.basename(r) == "ai-training"), None)
if src:                                   # Kaggle auto-extracts uploaded zips
    shutil.copytree(src, "/kaggle/working/ai-training", dirs_exist_ok=True)
else:
    zips = sorted(glob.glob("/kaggle/input/**/voice-enhancer-m3*.zip", recursive=True))
    assert zips, "Add the code dataset (voice-enhancer-m3c.zip) as an input"
    subprocess.run(["unzip", "-q", "-o", zips[-1], "voice-enhancer/ai-training/*", "-d", "/kaggle/working/"], check=True)
    shutil.move("/kaggle/working/voice-enhancer/ai-training", "/kaggle/working/ai-training")
%cd /kaggle/working/ai-training
!ls

In [ ]:
# 2) fast downloads (aria2c) and Python packages
r = subprocess.run("apt-get -qq update && apt-get -qq install -y aria2 > /dev/null", shell=True)
print("aria2c:", shutil.which("aria2c") or "NOT installed (downloads will use the slower built-in downloader)")
if not shutil.which("ffmpeg"):
    subprocess.run("apt-get -qq install -y ffmpeg > /dev/null", shell=True)
!pip -q install soundfile tensorboard
# 3) GTCRN model code and pretrained weights
if not os.path.exists("/kaggle/working/gtcrn"):
    !git clone -q https://github.com/Xiaobin-Rong/gtcrn /kaggle/working/gtcrn
CKPT = (glob.glob("/kaggle/working/gtcrn/checkpoints/*dns3*") or [None])[0]
print("pretrained checkpoint:", CKPT)

In [ ]:
# 4) smoke test (about a minute): stops here if anything in the stack is broken
!python smoke_test.py --gtcrn-dir /kaggle/working/gtcrn

## Data
Downloads go to the big temporary disk (not saved with the notebook). LibriTTS-R speech (CC BY 4.0, needs attribution in the app) plus room responses and noises (OpenSLR SLR28, Apache 2.0; its MUSAN-derived noises have per-clip licenses).
Hashes are checked against the values recorded in `fetch_datasets.py`. `MAX_FILES = 3000` keeps the first run to about 10 hours of speech per split; set it to `None` for everything.

In [ ]:
DATA = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
free = shutil.disk_usage(DATA).free / 1e9
print(f"{DATA}: {free:.0f} GB free")
assert free > 30, "Not enough temporary disk. Fetch only libritts_r_dev_clean + libritts_r_test_clean + rirs_noises and train on those."
t = time.time()
!python fetch_datasets.py list | tail -1
!python fetch_datasets.py fetch libritts_r_train_clean_100 libritts_r_dev_clean libritts_r_test_clean rirs_noises --root {DATA}/datasets --delete-archive
print(f"download + unpack: {(time.time() - t) / 60:.1f} min")
!cp {DATA}/datasets/MANIFEST.json /kaggle/working/MANIFEST.json

In [ ]:
MAX_FILES = 3000     # None = use all files
t = time.time()
flag = f"--max-files {MAX_FILES}" if MAX_FILES else ""
!python prepare_data.py --speech {DATA}/datasets/libritts_r_train_clean_100:train {DATA}/datasets/libritts_r_dev_clean:val {DATA}/datasets/libritts_r_test_clean:test --noise {DATA}/datasets/rirs_noises --out {DATA}/prepared {flag}
print(f"prepare: {(time.time() - t) / 60:.1f} min")
for n in ("speech_train", "speech_val", "noise_train", "noise_val", "rir_train", "rir_val"):
    k = sum(1 for l in open(f"{DATA}/prepared/lists/{n}.txt") if l.strip())
    print(f"{n:13} {k:7,d}")
    assert k > 0, f"{n} list is empty: the data steps above did not finish. Re-run them in order."

## Train
Defaults: 12 epochs x 1000 steps x batch 16 (roughly 0.5-2 hours; data mixing on the CPU is the bottleneck, estimated from a timing test: about 38 ms per example on one core). Watch the lines
`val SI-SDR` (should rise) and `high-band energy vs clean target` (3-5k / 5-7k / 7-8k should stay near 0 dB; if they sink below about -6 dB the model is making the voice dull, stop and tell me).
The cell stops itself before `TIME_LIMIT_MIN` and can be re-run (or continued in a new session) without losing progress.

In [ ]:
EPOCHS, STEPS, BATCH, LR = 12, 1000, 16, 2e-4
TIME_LIMIT_MIN = 10 * 60          # Kaggle sessions end after about 12 h; leave margin
WORKERS = min(os.cpu_count() or 2, 4)
RUN = "/kaggle/working/runs/gtcrn_ft1"
os.makedirs(RUN, exist_ok=True)
if not os.path.exists(f"{RUN}/last.pt"):      # continue from an earlier session's output if one is attached
    prev = sorted(glob.glob("/kaggle/input/**/runs/gtcrn_ft1/last.pt", recursive=True))
    if prev:
        shutil.copy(prev[-1], f"{RUN}/last.pt"); print("resuming from", prev[-1])
        b = prev[-1].replace("last.pt", "best.pt")
        if os.path.exists(b): shutil.copy(b, f"{RUN}/best.pt")
resume = f"--resume {RUN}/last.pt" if os.path.exists(f"{RUN}/last.pt") else ""
init = f"--init {CKPT}" if CKPT and not resume else ""
!python train.py --lists {DATA}/prepared/lists --arch gtcrn --gtcrn-dir /kaggle/working/gtcrn {init} --preset farfield --epochs {EPOCHS} --steps-per-epoch {STEPS} --batch {BATCH} --lr {LR} --workers {WORKERS} --time-limit-min {TIME_LIMIT_MIN} --out {RUN} {resume}
shutil.copy(f"{RUN}/best.pt", "/kaggle/working/best.pt")
print("saved /kaggle/working/best.pt (download it from the notebook's Output tab)")

## Try it on your test clips (skipped if you did not add any)

In [ ]:
clips = [p for p in glob.glob("/kaggle/input/*/*") if p.lower().endswith((".mp3", ".wav", ".mp4", ".m4a", ".flac")) and "voice-enhancer" not in p]
os.makedirs("/kaggle/working/enhanced", exist_ok=True)
for c in clips:
    out = f"/kaggle/working/enhanced/ft1_{os.path.splitext(os.path.basename(c))[0]}.wav"
    !python infer.py --ckpt {RUN}/best.pt --arch gtcrn --gtcrn-dir /kaggle/working/gtcrn --inp "{c}" --out "{out}"
    !python bench_clips.py --orig "{c}" --enh "{out}"
print("clips processed:", len(clips))